## Download full W&B step-by-step history

In [ ]:
# For downloading full step-by-step history from W&B
import wandb
import pandas as pd
from enum import StrEnum


class ProjectName(StrEnum):
    AEN_SAE_PYTHIA70M = "farischaudhry-imperial-college-london/aen-sae-llm-pythia70m"
    AEN_SAE_MECHANISTIC = "farischaudhry-imperial-college-london/aen-sae-mechanistic"
    AEN_SAE_MECHANISTIC_FINAL = "farischaudhry-imperial-college-london/aen-sae-mechanistic-final"
    AEN_SAE_LLAMA8B = "farischaudhry-imperial-college-london/aen-sae-llm-llama8b"

api = wandb.Api()

def download_wandb_history(project_name: ProjectName, save_path: str | None = None) -> pd.DataFrame:
    runs = api.runs(project_name)
    print(f"Found {len(runs)} total runs. Starting full history extraction...")

    all_history_rows = []
    runs_processed = 0

    for run in runs:
        # Skip crashed or currently running sweeps
        if run.state != "finished":
            continue

        print(f"Downloading history for run: {run.name} ({run.id})...")

        history_iterator = run.scan_history()
        for step_data in history_iterator:
            # Create a new row starting with the run identifiers
            clean_row = {
                "run_id": run.id,
                "run_name": run.name,
            }

            # Filter out histograms and complex objects (keep only numbers and strings)
            for key, value in step_data.items():
                # If the value is a standard flat type, keep it. 
                # If it's a dict or list (like a W&B histogram), ignore it.
                if isinstance(value, (int, float, str, bool)):
                    clean_row[key] = value
                    
            all_history_rows.append(clean_row)

        runs_processed += 1

    print(f"Finished processing {runs_processed} runs. Total history rows collected: {len(all_history_rows)}")#
    history_df = pd.DataFrame(all_history_rows)

    if 'step' in history_df.columns:
        history_df = history_df.sort_values(by=["run_name", "step"])

    if save_path is None:
        save_path = f"{project_name.split('/')[-1]}_step_by_step_history.csv"
    history_df.to_csv(save_path, index=False)
    print(f"History saved to {save_path}")
    return history_df

## Move/standardize checkpoints into final HF repo

In [ ]:
# from huggingface_hub import hf_hub_download, upload_file, list_repo_files
# import torch

# TARGET_REPO = "farischaudhry/adaptive-elastic-net-sae"
# REG_REPO = "farischaudhry/llm-llama8b-regularization-checkpoints"
# TOPK_REPO = "farischaudhry/llm-llama8b-topk-checkpoints"

# REG_FILE = "checkpoints/llama8b_regularization/seed0/adaptive_elastic_net/llm-adaptive_elastic_net-seed0.pt"
# REG_REVS = ["2eebdec", "070afa6", "83152b2", "3bebf6d", "68ffff5", "9ceddf7"]

# def label_from_metadata(ckpt):
#     rm = ckpt.get("run_metadata", {})
#     l1 = rm.get("hp_lambda_1")
#     l2 = rm.get("hp_lambda_2")
#     gamma = rm.get("hp_gamma")
#     if l1 is None:
#         return None
#     label = f"lambda1_{l1}_lambda2_{l2}_gamma_{gamma}"
#     return label.replace(".", "p")

# def topk_labels_from_metadata(ckpt):
#     rm = ckpt.get("run_metadata", {})
#     k = rm.get("hp_k")
#     seed = rm.get("seed")
#     run_name = ckpt.get("run_name")
#     if k is None:
#         return None, None, run_name
#     k_label = f"k{k}".replace(".", "p")
#     seed_label = f"seed{seed}" if seed is not None else "seed0"
#     return seed_label, k_label, run_name

# # Regularization checkpoints (including deleted revisions)
# for rev in REG_REVS:
#     local = hf_hub_download(repo_id=REG_REPO, filename=REG_FILE, revision=rev)
#     ckpt = torch.load(local, map_location="cpu")
#     label = label_from_metadata(ckpt) or rev
#     dest = f"checkpoints/llama8b/regularization/seed0/adaptive_elastic_net/{label}.pt"
#     upload_file(
#         path_or_fileobj=local,
#         path_in_repo=dest,
#         repo_id=TARGET_REPO,
#         repo_type="model",
#         commit_message=f"Add regularization checkpoint {label} from {rev}",
#     )
#     print("Uploaded:", dest)

# # TopK checkpoints
# def is_checkpoint(path):
#     return path.endswith((".pt", ".ckpt", ".bin", ".safetensors"))

# files = [f for f in list_repo_files(TOPK_REPO) if is_checkpoint(f)]
# for f in files:
#     local = hf_hub_download(repo_id=TOPK_REPO, filename=f)
#     seed_label = None
#     k_label = None
#     run_name = None
#     if f.endswith((".pt", ".ckpt", ".pth")):
#         ckpt = torch.load(local, map_location="cpu")
#         seed_label, k_label, run_name = topk_labels_from_metadata(ckpt)
#     if seed_label and k_label:
#         base_name = f"{k_label}_{run_name}" if run_name else k_label
#         dest = f"checkpoints/llama8b/topk/{seed_label}/{base_name}.pt"
#     else:
#         rel = f[len("checkpoints/"):] if f.startswith("checkpoints/") else f
#         dest = f"checkpoints/llama8b/topk/{rel}"
#     upload_file(
#         path_or_fileobj=local,
#         path_in_repo=dest,
#         repo_id=TARGET_REPO,
#         repo_type="model",
#         commit_message=f"Add topk checkpoint {k_label or f}",
#     )
#     print("Uploaded:", dest)

In [ ]:
# from huggingface_hub import upload_file
# from pathlib import Path

# TARGET_REPO = "farischaudhry/adaptive-elastic-net-sae"
# STEP_HISTORY_FILES = [
#     "pythia70m_step_by_step_history.csv",
#     "spiked_step_by_step_history.csv",
#     "llama8b_step_by_step_history.csv",
# ]

# for rel_path in STEP_HISTORY_FILES:
#     path = Path(rel_path)
#     if not path.exists():
#         print(f"Missing: {path}")
#         continue
#     upload_file(
#         path_or_fileobj=str(path),
#         path_in_repo=rel_path,
#         repo_id=TARGET_REPO,
#         repo_type="model",
#         commit_message=f"Add {rel_path}",
#     )
#     print("Uploaded:", rel_path)